# 0. Overview

This notebook computes DJF Pearson correlation over the Maritime Continent between Nino34 and these variables:
- rainfall
- 850 hPa wind vector components
- mean sea level pressure (MSLP)
- geopotential height at 850 hPa
- moisture-flux convergence
- streamfunction
- velocity potential

Each variable is shown for four periods:
- 1981-2025
- P1 = 1981-2006
- P2 = 2007-2025
- P2-P1 = difference between P2 and P1

The plotting domain is centered at 180 longitude and spans 60E-90W, 30S-30N.
Correlation statistics are computed on the same box with a 2 degree buffer on every side.
PNG outputs are written to `../../results/analisis_korelasi_26-19/`.


# 1. Import Library

In [ ]:
from pathlib import Path

# Define data paths (relative to notebook location)
RAINFALL_PATH = Path('../../external/ClimateData/mswep-monthly/mswep_monthly_combined.nc').resolve()
WIND_PATH = Path('../../external/ClimateData/era5-monthly/uv_850_global_1979-2025.nc').resolve()
MSLP_PATH = Path('../../external/ClimateData/era5-monthly/mslp_global_1979-2025.nc').resolve()
GEO850_PATH = Path('../../external/ClimateData/era5-monthly/geopotensial_850_global_1979-2025.nc').resolve()
MFC_PATH = Path('../../external/ClimateData/era5-monthly/viwve_viwvn_global_1979-2025.nc').resolve()
SVP_PATH = Path('../../external/ClimateData/era5-monthly/psi_chi_850_global_1979-2025.nc').resolve()
NINO34_PATH = Path('../../external/ClimateData/index-monthly/nino34.anom.csv').resolve()
GRAVITY = 9.80665

import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from scipy.ndimage import gaussian_filter
from scipy.stats import t as student_t, norm


# 2. Open Data

#### Open rainfall

In [ ]:
rainfall_path = RAINFALL_PATH
rainfall_chunks = {'time': 12}

ds_rain = xr.open_dataset(rainfall_path, chunks=rainfall_chunks)['precipitation']
ds_rain = ds_rain.assign_coords(lon=(ds_rain.lon % 360)).sortby('lon')


#### Open wind

In [ ]:
wind_path = WIND_PATH
wind_chunks = {'valid_time': 12}

ds_wind = xr.open_dataset(wind_path, chunks=wind_chunks)[['u', 'v']]
ds_wind = ds_wind.sel(pressure_level=850)
ds_wind = ds_wind.rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
ds_wind = ds_wind.assign_coords(lon=(ds_wind.lon % 360)).sortby('lon')


#### Open MSLP

In [ ]:
mslp_path = MSLP_PATH
mslp_chunks = {'valid_time': 12}

ds_mslp = xr.open_dataset(mslp_path, chunks=mslp_chunks)['msl']
ds_mslp = ds_mslp.rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
ds_mslp = ds_mslp.assign_coords(lon=(ds_mslp.lon % 360)).sortby('lon')

#### Open geopotential height 850 hPa

In [ ]:
gh850_path = GEO850_PATH
gh850_chunks = {'valid_time': 12}

ds_gh850 = xr.open_dataset(gh850_path, chunks=gh850_chunks)['z']
ds_gh850 = ds_gh850.sel(pressure_level=850)
ds_gh850 = ds_gh850.rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
ds_gh850 = ds_gh850.assign_coords(lon=(ds_gh850.lon % 360)).sortby('lon')
ds_gh850 = (ds_gh850 / GRAVITY).rename('gh850')
ds_gh850.attrs['units'] = 'm'
ds_gh850.attrs['standard_name'] = 'geopotential_height'
ds_gh850.attrs['long_name'] = 'Geopotential height at 850 hPa'

#### Open streamfunction / velocity potential

In [ ]:
svp_path = SVP_PATH

ds_svp = xr.open_dataset(svp_path)
if 'valid_time' in ds_svp.dims:
    ds_svp = ds_svp.chunk({'valid_time': 12}).rename({'valid_time': 'time'})
elif 'time' in ds_svp.dims:
    ds_svp = ds_svp.chunk({'time': 12})

if 'latitude' in ds_svp.dims or 'latitude' in ds_svp.coords:
    ds_svp = ds_svp.rename({'latitude': 'lat'})
if 'longitude' in ds_svp.dims or 'longitude' in ds_svp.coords:
    ds_svp = ds_svp.rename({'longitude': 'lon'})

ds_svp = ds_svp.assign_coords(lon=(ds_svp.lon % 360)).sortby('lon')

svp_var_map = {}
for target in ['psi', 'chi', 'u_psi', 'v_psi', 'u_chi', 'v_chi']:
    target_key = target.replace('_', '')
    matches = [
        name for name in ds_svp.data_vars
        if name.lower() == target or name.lower().replace('_', '') == target_key
    ]
    if not matches:
        matches = [
            name for name in ds_svp.data_vars
            if target_key in name.lower().replace('_', '')
        ]
    if not matches:
        raise KeyError(f'Variable for {target} not found in {svp_path}')
    svp_var_map[target] = matches[0]

psi_raw = ds_svp[svp_var_map['psi']]
chi_raw = ds_svp[svp_var_map['chi']]
u_psi_raw = ds_svp[svp_var_map['u_psi']]
v_psi_raw = ds_svp[svp_var_map['v_psi']]
u_chi_raw = ds_svp[svp_var_map['u_chi']]
v_chi_raw = ds_svp[svp_var_map['v_chi']]


#### Open moisture flux

In [ ]:
mfc_path = MFC_PATH
mfc_chunks = {'valid_time': 12}

ds_mfc = xr.open_dataset(mfc_path, chunks=mfc_chunks)[['viwve', 'viwvn']]
ds_mfc = ds_mfc.rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
ds_mfc = ds_mfc.assign_coords(lon=(ds_mfc.lon % 360)).sortby('lon')


#### Open Niño3.4 index

In [ ]:
nino34_path = NINO34_PATH
nino34_column = '   Nino Anom 3.4 Index  using ersstv5 from CPC  missing value -99.99 https://psl.noaa.gov/data/timeseries/month/'

df_nino34 = pd.read_csv(
    nino34_path,
    usecols=['Date', nino34_column],
    parse_dates=['Date'],
)


# 3. Pre-process

In [ ]:
full_years = np.arange(1981, 2026)
past_years = np.arange(1981, 2007)
recent_years = np.arange(2007, 2026)
analysis_start = '1980-12-01'
analysis_end = '2025-02-28'
mc_extent = [60, 270, -30, 30]
mc_xticks = np.arange(60, 271, 30)
mc_yticks = np.arange(-30, 31, 10)
mc_lat = slice(32, -32)
mc_lon = slice(58, 272)


In [ ]:
# Rainfall: select DJF months, assign DJF year, then crop to the Maritime Continent.
rain_mc = ds_rain.sel(
    time=slice(analysis_start, analysis_end),
    lat=mc_lat,
    lon=mc_lon,
)
rain_month_mask = rain_mc.time.dt.month.isin([12, 1, 2])
rain_djf_year = xr.where(rain_mc.time.dt.month == 12, rain_mc.time.dt.year + 1, rain_mc.time.dt.year)
rain_mc = rain_mc.sel(time=rain_month_mask).assign_coords(djf_year=('time', rain_djf_year.sel(time=rain_month_mask).data))
rain_mc = rain_mc.sel(time=rain_mc.djf_year.isin(full_years), lat=mc_lat, lon=mc_lon)
rain_clim = rain_mc
rain_past = rain_clim.sel(time=rain_clim.djf_year.isin(past_years))
rain_recent = rain_clim.sel(time=rain_clim.djf_year.isin(recent_years))

# Mean sea level pressure: same DJF treatment, then crop to the buffered analysis domain.
mslp_mc = ds_mslp.sel(
    time=slice(analysis_start, analysis_end),
    lat=mc_lat,
    lon=mc_lon,
)
mslp_month_mask = mslp_mc.time.dt.month.isin([12, 1, 2])
mslp_djf_year = xr.where(mslp_mc.time.dt.month == 12, mslp_mc.time.dt.year + 1, mslp_mc.time.dt.year)
mslp_mc = mslp_mc.sel(time=mslp_month_mask).assign_coords(djf_year=('time', mslp_djf_year.sel(time=mslp_month_mask).data))
mslp_mc = mslp_mc.sel(time=mslp_mc.djf_year.isin(full_years), lat=mc_lat, lon=mc_lon)
mslp_clim = mslp_mc
mslp_past = mslp_clim.sel(time=mslp_clim.djf_year.isin(past_years))
mslp_recent = mslp_clim.sel(time=mslp_clim.djf_year.isin(recent_years))

# Geopotential height at 850 hPa: same DJF treatment, then crop to the buffered analysis domain.
gh850_mc = ds_gh850.sel(
    time=slice(analysis_start, analysis_end),
    lat=mc_lat,
    lon=mc_lon,
)
gh850_month_mask = gh850_mc.time.dt.month.isin([12, 1, 2])
gh850_djf_year = xr.where(gh850_mc.time.dt.month == 12, gh850_mc.time.dt.year + 1, gh850_mc.time.dt.year)
gh850_mc = gh850_mc.sel(time=gh850_month_mask).assign_coords(djf_year=('time', gh850_djf_year.sel(time=gh850_month_mask).data))
gh850_mc = gh850_mc.sel(time=gh850_mc.djf_year.isin(full_years), lat=mc_lat, lon=mc_lon)
gh850_clim = gh850_mc
gh850_past = gh850_clim.sel(time=gh850_clim.djf_year.isin(past_years))
gh850_recent = gh850_clim.sel(time=gh850_clim.djf_year.isin(recent_years))

# Wind: same DJF treatment, then crop to the Maritime Continent.
wind_u_mc = ds_wind['u'].sel(
    time=slice(analysis_start, analysis_end),
    lat=mc_lat,
    lon=mc_lon,
)
wind_v_mc = ds_wind['v'].sel(
    time=slice(analysis_start, analysis_end),
    lat=mc_lat,
    lon=mc_lon,
)
wind_month_mask = wind_u_mc.time.dt.month.isin([12, 1, 2])
wind_djf_year = xr.where(wind_u_mc.time.dt.month == 12, wind_u_mc.time.dt.year + 1, wind_u_mc.time.dt.year)
wind_u_mc = wind_u_mc.sel(time=wind_month_mask).assign_coords(djf_year=('time', wind_djf_year.sel(time=wind_month_mask).data))
wind_v_mc = wind_v_mc.sel(time=wind_month_mask).assign_coords(djf_year=('time', wind_djf_year.sel(time=wind_month_mask).data))
wind_u_mc = wind_u_mc.sel(time=wind_u_mc.djf_year.isin(full_years), lat=mc_lat, lon=mc_lon)
wind_v_mc = wind_v_mc.sel(time=wind_v_mc.djf_year.isin(full_years), lat=mc_lat, lon=mc_lon)
u_clim = wind_u_mc
u_past = u_clim.sel(time=u_clim.djf_year.isin(past_years))
u_recent = u_clim.sel(time=u_clim.djf_year.isin(recent_years))
v_clim = wind_v_mc
v_past = v_clim.sel(time=v_clim.djf_year.isin(past_years))
v_recent = v_clim.sel(time=v_clim.djf_year.isin(recent_years))

# Streamfunction / velocity potential products: same DJF treatment, then crop to the Maritime Continent.
ds_svp_mc = xr.Dataset({
    'psi': psi_raw,
    'chi': chi_raw,
    'u_psi': u_psi_raw,
    'v_psi': v_psi_raw,
    'u_chi': u_chi_raw,
    'v_chi': v_chi_raw,
}).sel(
    time=slice(analysis_start, analysis_end),
    lat=mc_lat,
    lon=mc_lon,
)
svp_month_mask = ds_svp_mc.time.dt.month.isin([12, 1, 2])
svp_djf_year = xr.where(ds_svp_mc.time.dt.month == 12, ds_svp_mc.time.dt.year + 1, ds_svp_mc.time.dt.year)
ds_svp_mc = ds_svp_mc.sel(time=svp_month_mask).assign_coords(djf_year=('time', svp_djf_year.sel(time=svp_month_mask).data))
ds_svp_mc = ds_svp_mc.sel(time=ds_svp_mc.djf_year.isin(full_years), lat=mc_lat, lon=mc_lon)
psi_clim = ds_svp_mc['psi']
psi_past = psi_clim.sel(time=psi_clim.djf_year.isin(past_years))
psi_recent = psi_clim.sel(time=psi_clim.djf_year.isin(recent_years))
chi_clim = ds_svp_mc['chi']
chi_past = chi_clim.sel(time=chi_clim.djf_year.isin(past_years))
chi_recent = chi_clim.sel(time=chi_clim.djf_year.isin(recent_years))
u_psi_clim = ds_svp_mc['u_psi']
u_psi_past = u_psi_clim.sel(time=u_psi_clim.djf_year.isin(past_years))
u_psi_recent = u_psi_clim.sel(time=u_psi_clim.djf_year.isin(recent_years))
v_psi_clim = ds_svp_mc['v_psi']
v_psi_past = v_psi_clim.sel(time=v_psi_clim.djf_year.isin(past_years))
v_psi_recent = v_psi_clim.sel(time=v_psi_clim.djf_year.isin(recent_years))
u_chi_clim = ds_svp_mc['u_chi']
u_chi_past = u_chi_clim.sel(time=u_chi_clim.djf_year.isin(past_years))
u_chi_recent = u_chi_clim.sel(time=u_chi_clim.djf_year.isin(recent_years))
v_chi_clim = ds_svp_mc['v_chi']
v_chi_past = v_chi_clim.sel(time=v_chi_clim.djf_year.isin(past_years))
v_chi_recent = v_chi_clim.sel(time=v_chi_clim.djf_year.isin(recent_years))

# Moisture flux convergence: compute the derivative on the broader field first, then crop to the Maritime Continent.
mfc_qx_mc = ds_mfc['viwve'].sel(
    time=slice(analysis_start, analysis_end),
    lat=mc_lat,
    lon=mc_lon,
)
mfc_qy_mc = ds_mfc['viwvn'].sel(
    time=slice(analysis_start, analysis_end),
    lat=mc_lat,
    lon=mc_lon,
)
mfc_month_mask = mfc_qx_mc.time.dt.month.isin([12, 1, 2])
mfc_djf_year = xr.where(mfc_qx_mc.time.dt.month == 12, mfc_qx_mc.time.dt.year + 1, mfc_qx_mc.time.dt.year)
mfc_qx_mc = mfc_qx_mc.sel(time=mfc_month_mask).assign_coords(djf_year=('time', mfc_djf_year.sel(time=mfc_month_mask).data))
mfc_qy_mc = mfc_qy_mc.sel(time=mfc_month_mask).assign_coords(djf_year=('time', mfc_djf_year.sel(time=mfc_month_mask).data))
qx_clim = mfc_qx_mc.sel(time=mfc_qx_mc.djf_year.isin(full_years), lat=mc_lat, lon=mc_lon)
qx_past = qx_clim.sel(time=qx_clim.djf_year.isin(past_years))
qx_recent = qx_clim.sel(time=qx_clim.djf_year.isin(recent_years))
qy_clim = mfc_qy_mc.sel(time=mfc_qy_mc.djf_year.isin(full_years), lat=mc_lat, lon=mc_lon)
qy_past = qy_clim.sel(time=qy_clim.djf_year.isin(past_years))
qy_recent = qy_clim.sel(time=qy_clim.djf_year.isin(recent_years))

a = 6.371e6
phi = np.deg2rad(mfc_qx_mc['lat'])
dqx_dlam = mfc_qx_mc.differentiate('lon') * (180.0 / np.pi)
dqycos_dphi = (mfc_qy_mc * np.cos(phi)).differentiate('lat') * (180.0 / np.pi)
div_q = (1.0 / (a * np.cos(phi))) * dqx_dlam + (1.0 / (a * np.cos(phi))) * dqycos_dphi
mfc_mc = (-div_q).rename('mfc')
mfc_mc.attrs['units'] = 'kg m^-2 s^-1'
mfc_mc.attrs['long_name'] = 'moisture flux convergence'
mfc_mc = mfc_mc.sel(time=mfc_mc.djf_year.isin(full_years), lat=mc_lat, lon=mc_lon)
mfc_clim = mfc_mc
mfc_past = mfc_clim.sel(time=mfc_clim.djf_year.isin(past_years))
mfc_recent = mfc_clim.sel(time=mfc_clim.djf_year.isin(recent_years))

# Niño3.4: keep only DJF seasons that match the analysis years.
nino34_mc = df_nino34.set_index('Date').loc['1980-12-01':'2025-02-28']
nino34_mc = nino34_mc[nino34_mc.index.month.isin([12, 1, 2])].copy()
nino34_mc['djf_year'] = nino34_mc.index.year + (nino34_mc.index.month == 12).astype('int8')
nino34_mc = nino34_mc[nino34_mc['djf_year'].isin(full_years)].copy()
nino34_clim = nino34_mc
nino34_past = nino34_clim[nino34_clim['djf_year'].isin(past_years)].copy()
nino34_recent = nino34_clim[nino34_clim['djf_year'].isin(recent_years)].copy()


In [ ]:
rain_clim_djf_corr = rain_clim.groupby('djf_year').mean('time')
rain_past_djf_corr = rain_past.groupby('djf_year').mean('time')
rain_recent_djf_corr = rain_recent.groupby('djf_year').mean('time')

u_clim_djf_corr = u_clim.groupby('djf_year').mean('time')
u_past_djf_corr = u_past.groupby('djf_year').mean('time')
u_recent_djf_corr = u_recent.groupby('djf_year').mean('time')

v_clim_djf_corr = v_clim.groupby('djf_year').mean('time')
v_past_djf_corr = v_past.groupby('djf_year').mean('time')
v_recent_djf_corr = v_recent.groupby('djf_year').mean('time')

psi_clim_djf_corr = psi_clim.groupby('djf_year').mean('time')
psi_past_djf_corr = psi_past.groupby('djf_year').mean('time')
psi_recent_djf_corr = psi_recent.groupby('djf_year').mean('time')

chi_clim_djf_corr = chi_clim.groupby('djf_year').mean('time')
chi_past_djf_corr = chi_past.groupby('djf_year').mean('time')
chi_recent_djf_corr = chi_recent.groupby('djf_year').mean('time')

u_psi_clim_djf_corr = u_psi_clim.groupby('djf_year').mean('time')
u_psi_past_djf_corr = u_psi_past.groupby('djf_year').mean('time')
u_psi_recent_djf_corr = u_psi_recent.groupby('djf_year').mean('time')

v_psi_clim_djf_corr = v_psi_clim.groupby('djf_year').mean('time')
v_psi_past_djf_corr = v_psi_past.groupby('djf_year').mean('time')
v_psi_recent_djf_corr = v_psi_recent.groupby('djf_year').mean('time')

u_chi_clim_djf_corr = u_chi_clim.groupby('djf_year').mean('time')
u_chi_past_djf_corr = u_chi_past.groupby('djf_year').mean('time')
u_chi_recent_djf_corr = u_chi_recent.groupby('djf_year').mean('time')

v_chi_clim_djf_corr = v_chi_clim.groupby('djf_year').mean('time')
v_chi_past_djf_corr = v_chi_past.groupby('djf_year').mean('time')
v_chi_recent_djf_corr = v_chi_recent.groupby('djf_year').mean('time')

mfc_clim_djf_corr = mfc_clim.groupby('djf_year').mean('time')
mfc_past_djf_corr = mfc_past.groupby('djf_year').mean('time')
mfc_recent_djf_corr = mfc_recent.groupby('djf_year').mean('time')

qx_clim_djf_corr = qx_clim.groupby('djf_year').mean('time')
qx_past_djf_corr = qx_past.groupby('djf_year').mean('time')
qx_recent_djf_corr = qx_recent.groupby('djf_year').mean('time')

qy_clim_djf_corr = qy_clim.groupby('djf_year').mean('time')
qy_past_djf_corr = qy_past.groupby('djf_year').mean('time')
qy_recent_djf_corr = qy_recent.groupby('djf_year').mean('time')

mslp_clim_djf_corr = mslp_clim.groupby('djf_year').mean('time')
mslp_past_djf_corr = mslp_past.groupby('djf_year').mean('time')
mslp_recent_djf_corr = mslp_recent.groupby('djf_year').mean('time')

gh850_clim_djf_corr = gh850_clim.groupby('djf_year').mean('time')
gh850_past_djf_corr = gh850_past.groupby('djf_year').mean('time')
gh850_recent_djf_corr = gh850_recent.groupby('djf_year').mean('time')

nino34_clim_djf_corr_series = nino34_clim.groupby('djf_year')[nino34_column].mean()
nino34_past_djf_corr_series = nino34_past.groupby('djf_year')[nino34_column].mean()
nino34_recent_djf_corr_series = nino34_recent.groupby('djf_year')[nino34_column].mean()

nino34_clim_djf_corr = xr.DataArray(
    nino34_clim_djf_corr_series.to_numpy(),
    coords={'djf_year': nino34_clim_djf_corr_series.index.to_numpy()},
    dims='djf_year',
    name='nino34',
)
nino34_past_djf_corr = xr.DataArray(
    nino34_past_djf_corr_series.to_numpy(),
    coords={'djf_year': nino34_past_djf_corr_series.index.to_numpy()},
    dims='djf_year',
    name='nino34',
)
nino34_recent_djf_corr = xr.DataArray(
    nino34_recent_djf_corr_series.to_numpy(),
    coords={'djf_year': nino34_recent_djf_corr_series.index.to_numpy()},
    dims='djf_year',
    name='nino34',
)

print('Rainfall DJF seasons:', int(rain_clim_djf_corr.sizes['djf_year']))
print('Wind DJF seasons:', int(u_clim_djf_corr.sizes['djf_year']))
print('Streamfunction DJF seasons:', int(psi_clim_djf_corr.sizes['djf_year']))
print('Velocity potential DJF seasons:', int(chi_clim_djf_corr.sizes['djf_year']))
print('MFC DJF seasons:', int(mfc_clim_djf_corr.sizes['djf_year']))
print('MSLP DJF seasons:', int(mslp_clim_djf_corr.sizes['djf_year']))
print('Geopotential height 850 hPa DJF seasons:', int(gh850_clim_djf_corr.sizes['djf_year']))
print('Niño3.4 DJF seasons:', int(nino34_clim_djf_corr.sizes['djf_year']))


# 4. Correlation & Significance

Scalar fields use p < 0.05 stippling. Vector significance is computed componentwise and combined with OR.

In [ ]:
rain_nino34_clim_djf_corr = nino34_clim_djf_corr.copy()
rain_nino34_past_djf_corr = nino34_past_djf_corr.copy()
rain_nino34_recent_djf_corr = nino34_recent_djf_corr.copy()

rain_clim_djf_corr, rain_nino34_clim_djf_corr = xr.align(rain_clim_djf_corr, rain_nino34_clim_djf_corr, join='inner')
rain_past_djf_corr, rain_nino34_past_djf_corr = xr.align(rain_past_djf_corr, rain_nino34_past_djf_corr, join='inner')
rain_recent_djf_corr, rain_nino34_recent_djf_corr = xr.align(rain_recent_djf_corr, rain_nino34_recent_djf_corr, join='inner')

rain_corr_clim = xr.corr(rain_clim_djf_corr, rain_nino34_clim_djf_corr, dim='djf_year').compute()
rain_corr_past = xr.corr(rain_past_djf_corr, rain_nino34_past_djf_corr, dim='djf_year').compute()
rain_corr_recent = xr.corr(rain_recent_djf_corr, rain_nino34_recent_djf_corr, dim='djf_year').compute()
rain_corr_recent_minus_past = rain_corr_recent - rain_corr_past

rain_n_full = int(rain_clim_djf_corr.sizes['djf_year'])
rain_n_past = int(rain_past_djf_corr.sizes['djf_year'])
rain_n_recent = int(rain_recent_djf_corr.sizes['djf_year'])

rain_corr_clim_safe = xr.where(np.abs(rain_corr_clim) >= 0.999999, np.sign(rain_corr_clim) * 0.999999, rain_corr_clim)
rain_corr_past_safe = xr.where(np.abs(rain_corr_past) >= 0.999999, np.sign(rain_corr_past) * 0.999999, rain_corr_past)
rain_corr_recent_safe = xr.where(np.abs(rain_corr_recent) >= 0.999999, np.sign(rain_corr_recent) * 0.999999, rain_corr_recent)

rain_corr_clim_t = rain_corr_clim_safe * np.sqrt((rain_n_full - 2) / (1 - rain_corr_clim_safe ** 2))
rain_corr_past_t = rain_corr_past_safe * np.sqrt((rain_n_past - 2) / (1 - rain_corr_past_safe ** 2))
rain_corr_recent_t = rain_corr_recent_safe * np.sqrt((rain_n_recent - 2) / (1 - rain_corr_recent_safe ** 2))

rain_corr_clim_p = xr.apply_ufunc(
    lambda x: 2 * student_t.sf(np.abs(x), df=rain_n_full - 2),
    rain_corr_clim_t,
    vectorize=True,
    dask='allowed',
    output_dtypes=[float],
).compute()
rain_corr_past_p = xr.apply_ufunc(
    lambda x: 2 * student_t.sf(np.abs(x), df=rain_n_past - 2),
    rain_corr_past_t,
    vectorize=True,
    dask='allowed',
    output_dtypes=[float],
).compute()
rain_corr_recent_p = xr.apply_ufunc(
    lambda x: 2 * student_t.sf(np.abs(x), df=rain_n_recent - 2),
    rain_corr_recent_t,
    vectorize=True,
    dask='allowed',
    output_dtypes=[float],
).compute()

rain_z_stat = (np.arctanh(rain_corr_recent_safe) - np.arctanh(rain_corr_past_safe)) / np.sqrt(1 / (rain_n_recent - 3) + 1 / (rain_n_past - 3))
rain_corr_recent_minus_past_p = xr.apply_ufunc(
    lambda x: 2 * norm.sf(np.abs(x)),
    rain_z_stat,
    vectorize=True,
    dask='allowed',
    output_dtypes=[float],
).compute()

rain_corr_clim_sig = rain_corr_clim_p < 0.05
rain_corr_past_sig = rain_corr_past_p < 0.05
rain_corr_recent_sig = rain_corr_recent_p < 0.05
rain_corr_recent_minus_past_sig = rain_corr_recent_minus_past_p < 0.05


In [ ]:
wind_nino34_clim_djf_corr = nino34_clim_djf_corr.copy()
wind_nino34_past_djf_corr = nino34_past_djf_corr.copy()
wind_nino34_recent_djf_corr = nino34_recent_djf_corr.copy()

u_clim_djf_corr, wind_nino34_clim_djf_corr = xr.align(u_clim_djf_corr, wind_nino34_clim_djf_corr, join='inner')
u_past_djf_corr, wind_nino34_past_djf_corr = xr.align(u_past_djf_corr, wind_nino34_past_djf_corr, join='inner')
u_recent_djf_corr, wind_nino34_recent_djf_corr = xr.align(u_recent_djf_corr, wind_nino34_recent_djf_corr, join='inner')

v_clim_djf_corr, _ = xr.align(v_clim_djf_corr, wind_nino34_clim_djf_corr, join='inner')
v_past_djf_corr, _ = xr.align(v_past_djf_corr, wind_nino34_past_djf_corr, join='inner')
v_recent_djf_corr, _ = xr.align(v_recent_djf_corr, wind_nino34_recent_djf_corr, join='inner')

wind_u_corr_clim = xr.corr(u_clim_djf_corr, wind_nino34_clim_djf_corr, dim='djf_year').compute()
wind_u_corr_past = xr.corr(u_past_djf_corr, wind_nino34_past_djf_corr, dim='djf_year').compute()
wind_u_corr_recent = xr.corr(u_recent_djf_corr, wind_nino34_recent_djf_corr, dim='djf_year').compute()
wind_u_corr_recent_minus_past = wind_u_corr_recent - wind_u_corr_past

wind_v_corr_clim = xr.corr(v_clim_djf_corr, wind_nino34_clim_djf_corr, dim='djf_year').compute()
wind_v_corr_past = xr.corr(v_past_djf_corr, wind_nino34_past_djf_corr, dim='djf_year').compute()
wind_v_corr_recent = xr.corr(v_recent_djf_corr, wind_nino34_recent_djf_corr, dim='djf_year').compute()
wind_v_corr_recent_minus_past = wind_v_corr_recent - wind_v_corr_past

wind_n_full = int(u_clim_djf_corr.sizes['djf_year'])
wind_n_past = int(u_past_djf_corr.sizes['djf_year'])
wind_n_recent = int(u_recent_djf_corr.sizes['djf_year'])

wind_u_corr_clim_safe = xr.where(np.abs(wind_u_corr_clim) >= 0.999999, np.sign(wind_u_corr_clim) * 0.999999, wind_u_corr_clim)
wind_u_corr_past_safe = xr.where(np.abs(wind_u_corr_past) >= 0.999999, np.sign(wind_u_corr_past) * 0.999999, wind_u_corr_past)
wind_u_corr_recent_safe = xr.where(np.abs(wind_u_corr_recent) >= 0.999999, np.sign(wind_u_corr_recent) * 0.999999, wind_u_corr_recent)
wind_v_corr_clim_safe = xr.where(np.abs(wind_v_corr_clim) >= 0.999999, np.sign(wind_v_corr_clim) * 0.999999, wind_v_corr_clim)
wind_v_corr_past_safe = xr.where(np.abs(wind_v_corr_past) >= 0.999999, np.sign(wind_v_corr_past) * 0.999999, wind_v_corr_past)
wind_v_corr_recent_safe = xr.where(np.abs(wind_v_corr_recent) >= 0.999999, np.sign(wind_v_corr_recent) * 0.999999, wind_v_corr_recent)

wind_u_corr_clim_t = wind_u_corr_clim_safe * np.sqrt((wind_n_full - 2) / (1 - wind_u_corr_clim_safe ** 2))
wind_u_corr_past_t = wind_u_corr_past_safe * np.sqrt((wind_n_past - 2) / (1 - wind_u_corr_past_safe ** 2))
wind_u_corr_recent_t = wind_u_corr_recent_safe * np.sqrt((wind_n_recent - 2) / (1 - wind_u_corr_recent_safe ** 2))
wind_v_corr_clim_t = wind_v_corr_clim_safe * np.sqrt((wind_n_full - 2) / (1 - wind_v_corr_clim_safe ** 2))
wind_v_corr_past_t = wind_v_corr_past_safe * np.sqrt((wind_n_past - 2) / (1 - wind_v_corr_past_safe ** 2))
wind_v_corr_recent_t = wind_v_corr_recent_safe * np.sqrt((wind_n_recent - 2) / (1 - wind_v_corr_recent_safe ** 2))

wind_u_corr_clim_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=wind_n_full - 2), wind_u_corr_clim_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
wind_u_corr_past_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=wind_n_past - 2), wind_u_corr_past_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
wind_u_corr_recent_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=wind_n_recent - 2), wind_u_corr_recent_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
wind_v_corr_clim_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=wind_n_full - 2), wind_v_corr_clim_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
wind_v_corr_past_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=wind_n_past - 2), wind_v_corr_past_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
wind_v_corr_recent_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=wind_n_recent - 2), wind_v_corr_recent_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()

wind_vector_sig_clim = (wind_u_corr_clim_p < 0.05) | (wind_v_corr_clim_p < 0.05)
wind_vector_sig_past = (wind_u_corr_past_p < 0.05) | (wind_v_corr_past_p < 0.05)
wind_vector_sig_recent = (wind_u_corr_recent_p < 0.05) | (wind_v_corr_recent_p < 0.05)
wind_vector_sig_recent_minus_past = (wind_u_corr_recent_p < 0.05) | (wind_v_corr_recent_p < 0.05)


In [ ]:
mfc_nino34_clim_djf_corr = nino34_clim_djf_corr.copy()
mfc_nino34_past_djf_corr = nino34_past_djf_corr.copy()
mfc_nino34_recent_djf_corr = nino34_recent_djf_corr.copy()

mfc_clim_djf_corr, mfc_nino34_clim_djf_corr = xr.align(mfc_clim_djf_corr, mfc_nino34_clim_djf_corr, join='inner')
mfc_past_djf_corr, mfc_nino34_past_djf_corr = xr.align(mfc_past_djf_corr, mfc_nino34_past_djf_corr, join='inner')
mfc_recent_djf_corr, mfc_nino34_recent_djf_corr = xr.align(mfc_recent_djf_corr, mfc_nino34_recent_djf_corr, join='inner')

qx_clim_djf_corr, _ = xr.align(qx_clim_djf_corr, mfc_nino34_clim_djf_corr, join='inner')
qx_past_djf_corr, _ = xr.align(qx_past_djf_corr, mfc_nino34_past_djf_corr, join='inner')
qx_recent_djf_corr, _ = xr.align(qx_recent_djf_corr, mfc_nino34_recent_djf_corr, join='inner')

qy_clim_djf_corr, _ = xr.align(qy_clim_djf_corr, mfc_nino34_clim_djf_corr, join='inner')
qy_past_djf_corr, _ = xr.align(qy_past_djf_corr, mfc_nino34_past_djf_corr, join='inner')
qy_recent_djf_corr, _ = xr.align(qy_recent_djf_corr, mfc_nino34_recent_djf_corr, join='inner')

mfc_corr_clim = xr.corr(mfc_clim_djf_corr, mfc_nino34_clim_djf_corr, dim='djf_year').compute()
mfc_corr_past = xr.corr(mfc_past_djf_corr, mfc_nino34_past_djf_corr, dim='djf_year').compute()
mfc_corr_recent = xr.corr(mfc_recent_djf_corr, mfc_nino34_recent_djf_corr, dim='djf_year').compute()
mfc_corr_recent_minus_past = mfc_corr_recent - mfc_corr_past

mfc_qx_corr_clim = xr.corr(qx_clim_djf_corr, mfc_nino34_clim_djf_corr, dim='djf_year').compute()
mfc_qx_corr_past = xr.corr(qx_past_djf_corr, mfc_nino34_past_djf_corr, dim='djf_year').compute()
mfc_qx_corr_recent = xr.corr(qx_recent_djf_corr, mfc_nino34_recent_djf_corr, dim='djf_year').compute()
mfc_qx_corr_recent_minus_past = mfc_qx_corr_recent - mfc_qx_corr_past

mfc_qy_corr_clim = xr.corr(qy_clim_djf_corr, mfc_nino34_clim_djf_corr, dim='djf_year').compute()
mfc_qy_corr_past = xr.corr(qy_past_djf_corr, mfc_nino34_past_djf_corr, dim='djf_year').compute()
mfc_qy_corr_recent = xr.corr(qy_recent_djf_corr, mfc_nino34_recent_djf_corr, dim='djf_year').compute()
mfc_qy_corr_recent_minus_past = mfc_qy_corr_recent - mfc_qy_corr_past

mfc_n_full = int(mfc_clim_djf_corr.sizes['djf_year'])
mfc_n_past = int(mfc_past_djf_corr.sizes['djf_year'])
mfc_n_recent = int(mfc_recent_djf_corr.sizes['djf_year'])

mfc_corr_clim_safe = xr.where(np.abs(mfc_corr_clim) >= 0.999999, np.sign(mfc_corr_clim) * 0.999999, mfc_corr_clim)
mfc_corr_past_safe = xr.where(np.abs(mfc_corr_past) >= 0.999999, np.sign(mfc_corr_past) * 0.999999, mfc_corr_past)
mfc_corr_recent_safe = xr.where(np.abs(mfc_corr_recent) >= 0.999999, np.sign(mfc_corr_recent) * 0.999999, mfc_corr_recent)

mfc_corr_clim_t = mfc_corr_clim_safe * np.sqrt((mfc_n_full - 2) / (1 - mfc_corr_clim_safe ** 2))
mfc_corr_past_t = mfc_corr_past_safe * np.sqrt((mfc_n_past - 2) / (1 - mfc_corr_past_safe ** 2))
mfc_corr_recent_t = mfc_corr_recent_safe * np.sqrt((mfc_n_recent - 2) / (1 - mfc_corr_recent_safe ** 2))

mfc_corr_clim_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=mfc_n_full - 2), mfc_corr_clim_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
mfc_corr_past_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=mfc_n_past - 2), mfc_corr_past_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
mfc_corr_recent_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=mfc_n_recent - 2), mfc_corr_recent_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()

mfc_z_stat = (np.arctanh(mfc_corr_recent_safe) - np.arctanh(mfc_corr_past_safe)) / np.sqrt(1 / (mfc_n_recent - 3) + 1 / (mfc_n_past - 3))
mfc_corr_recent_minus_past_p = xr.apply_ufunc(lambda x: 2 * norm.sf(np.abs(x)), mfc_z_stat, vectorize=True, dask='allowed', output_dtypes=[float]).compute()

mfc_corr_clim_sig = mfc_corr_clim_p < 0.05
mfc_corr_past_sig = mfc_corr_past_p < 0.05
mfc_corr_recent_sig = mfc_corr_recent_p < 0.05
mfc_corr_recent_minus_past_sig = mfc_corr_recent_minus_past_p < 0.05


In [ ]:
psi_nino34_clim_djf_corr = nino34_clim_djf_corr.copy()
psi_nino34_past_djf_corr = nino34_past_djf_corr.copy()
psi_nino34_recent_djf_corr = nino34_recent_djf_corr.copy()

psi_clim_djf_corr, psi_nino34_clim_djf_corr = xr.align(psi_clim_djf_corr, psi_nino34_clim_djf_corr, join='inner')
psi_past_djf_corr, psi_nino34_past_djf_corr = xr.align(psi_past_djf_corr, psi_nino34_past_djf_corr, join='inner')
psi_recent_djf_corr, psi_nino34_recent_djf_corr = xr.align(psi_recent_djf_corr, psi_nino34_recent_djf_corr, join='inner')

u_psi_clim_djf_corr, _ = xr.align(u_psi_clim_djf_corr, psi_nino34_clim_djf_corr, join='inner')
u_psi_past_djf_corr, _ = xr.align(u_psi_past_djf_corr, psi_nino34_past_djf_corr, join='inner')
u_psi_recent_djf_corr, _ = xr.align(u_psi_recent_djf_corr, psi_nino34_recent_djf_corr, join='inner')

v_psi_clim_djf_corr, _ = xr.align(v_psi_clim_djf_corr, psi_nino34_clim_djf_corr, join='inner')
v_psi_past_djf_corr, _ = xr.align(v_psi_past_djf_corr, psi_nino34_past_djf_corr, join='inner')
v_psi_recent_djf_corr, _ = xr.align(v_psi_recent_djf_corr, psi_nino34_recent_djf_corr, join='inner')

psi_corr_clim = xr.corr(psi_clim_djf_corr, psi_nino34_clim_djf_corr, dim='djf_year').compute()
psi_corr_past = xr.corr(psi_past_djf_corr, psi_nino34_past_djf_corr, dim='djf_year').compute()
psi_corr_recent = xr.corr(psi_recent_djf_corr, psi_nino34_recent_djf_corr, dim='djf_year').compute()
psi_corr_recent_minus_past = psi_corr_recent - psi_corr_past

u_psi_corr_clim = xr.corr(u_psi_clim_djf_corr, psi_nino34_clim_djf_corr, dim='djf_year').compute()
u_psi_corr_past = xr.corr(u_psi_past_djf_corr, psi_nino34_past_djf_corr, dim='djf_year').compute()
u_psi_corr_recent = xr.corr(u_psi_recent_djf_corr, psi_nino34_recent_djf_corr, dim='djf_year').compute()
u_psi_corr_recent_minus_past = u_psi_corr_recent - u_psi_corr_past

v_psi_corr_clim = xr.corr(v_psi_clim_djf_corr, psi_nino34_clim_djf_corr, dim='djf_year').compute()
v_psi_corr_past = xr.corr(v_psi_past_djf_corr, psi_nino34_past_djf_corr, dim='djf_year').compute()
v_psi_corr_recent = xr.corr(v_psi_recent_djf_corr, psi_nino34_recent_djf_corr, dim='djf_year').compute()
v_psi_corr_recent_minus_past = v_psi_corr_recent - v_psi_corr_past

psi_n_full = int(psi_clim_djf_corr.sizes['djf_year'])
psi_n_past = int(psi_past_djf_corr.sizes['djf_year'])
psi_n_recent = int(psi_recent_djf_corr.sizes['djf_year'])

psi_corr_clim_safe = xr.where(np.abs(psi_corr_clim) >= 0.999999, np.sign(psi_corr_clim) * 0.999999, psi_corr_clim)
psi_corr_past_safe = xr.where(np.abs(psi_corr_past) >= 0.999999, np.sign(psi_corr_past) * 0.999999, psi_corr_past)
psi_corr_recent_safe = xr.where(np.abs(psi_corr_recent) >= 0.999999, np.sign(psi_corr_recent) * 0.999999, psi_corr_recent)

psi_corr_clim_t = psi_corr_clim_safe * np.sqrt((psi_n_full - 2) / (1 - psi_corr_clim_safe ** 2))
psi_corr_past_t = psi_corr_past_safe * np.sqrt((psi_n_past - 2) / (1 - psi_corr_past_safe ** 2))
psi_corr_recent_t = psi_corr_recent_safe * np.sqrt((psi_n_recent - 2) / (1 - psi_corr_recent_safe ** 2))

psi_corr_clim_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=psi_n_full - 2), psi_corr_clim_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
psi_corr_past_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=psi_n_past - 2), psi_corr_past_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
psi_corr_recent_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=psi_n_recent - 2), psi_corr_recent_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()

psi_z_stat = (np.arctanh(psi_corr_recent_safe) - np.arctanh(psi_corr_past_safe)) / np.sqrt(1 / (psi_n_recent - 3) + 1 / (psi_n_past - 3))
psi_corr_recent_minus_past_p = xr.apply_ufunc(lambda x: 2 * norm.sf(np.abs(x)), psi_z_stat, vectorize=True, dask='allowed', output_dtypes=[float]).compute()

u_psi_corr_clim_safe = xr.where(np.abs(u_psi_corr_clim) >= 0.999999, np.sign(u_psi_corr_clim) * 0.999999, u_psi_corr_clim)
u_psi_corr_past_safe = xr.where(np.abs(u_psi_corr_past) >= 0.999999, np.sign(u_psi_corr_past) * 0.999999, u_psi_corr_past)
u_psi_corr_recent_safe = xr.where(np.abs(u_psi_corr_recent) >= 0.999999, np.sign(u_psi_corr_recent) * 0.999999, u_psi_corr_recent)
v_psi_corr_clim_safe = xr.where(np.abs(v_psi_corr_clim) >= 0.999999, np.sign(v_psi_corr_clim) * 0.999999, v_psi_corr_clim)
v_psi_corr_past_safe = xr.where(np.abs(v_psi_corr_past) >= 0.999999, np.sign(v_psi_corr_past) * 0.999999, v_psi_corr_past)
v_psi_corr_recent_safe = xr.where(np.abs(v_psi_corr_recent) >= 0.999999, np.sign(v_psi_corr_recent) * 0.999999, v_psi_corr_recent)

u_psi_corr_clim_t = u_psi_corr_clim_safe * np.sqrt((psi_n_full - 2) / (1 - u_psi_corr_clim_safe ** 2))
u_psi_corr_past_t = u_psi_corr_past_safe * np.sqrt((psi_n_past - 2) / (1 - u_psi_corr_past_safe ** 2))
u_psi_corr_recent_t = u_psi_corr_recent_safe * np.sqrt((psi_n_recent - 2) / (1 - u_psi_corr_recent_safe ** 2))
v_psi_corr_clim_t = v_psi_corr_clim_safe * np.sqrt((psi_n_full - 2) / (1 - v_psi_corr_clim_safe ** 2))
v_psi_corr_past_t = v_psi_corr_past_safe * np.sqrt((psi_n_past - 2) / (1 - v_psi_corr_past_safe ** 2))
v_psi_corr_recent_t = v_psi_corr_recent_safe * np.sqrt((psi_n_recent - 2) / (1 - v_psi_corr_recent_safe ** 2))

u_psi_corr_clim_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=psi_n_full - 2), u_psi_corr_clim_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
u_psi_corr_past_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=psi_n_past - 2), u_psi_corr_past_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
u_psi_corr_recent_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=psi_n_recent - 2), u_psi_corr_recent_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
v_psi_corr_clim_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=psi_n_full - 2), v_psi_corr_clim_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
v_psi_corr_past_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=psi_n_past - 2), v_psi_corr_past_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
v_psi_corr_recent_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=psi_n_recent - 2), v_psi_corr_recent_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()

psi_corr_clim_sig = psi_corr_clim_p < 0.05
psi_corr_past_sig = psi_corr_past_p < 0.05
psi_corr_recent_sig = psi_corr_recent_p < 0.05
psi_corr_recent_minus_past_sig = psi_corr_recent_minus_past_p < 0.05
psi_vector_sig_clim = (u_psi_corr_clim_p < 0.05) | (v_psi_corr_clim_p < 0.05)
psi_vector_sig_past = (u_psi_corr_past_p < 0.05) | (v_psi_corr_past_p < 0.05)
psi_vector_sig_recent = (u_psi_corr_recent_p < 0.05) | (v_psi_corr_recent_p < 0.05)
psi_vector_sig_recent_minus_past = (u_psi_corr_recent_p < 0.05) | (v_psi_corr_recent_p < 0.05)


In [ ]:
chi_nino34_clim_djf_corr = nino34_clim_djf_corr.copy()
chi_nino34_past_djf_corr = nino34_past_djf_corr.copy()
chi_nino34_recent_djf_corr = nino34_recent_djf_corr.copy()

chi_clim_djf_corr, chi_nino34_clim_djf_corr = xr.align(chi_clim_djf_corr, chi_nino34_clim_djf_corr, join='inner')
chi_past_djf_corr, chi_nino34_past_djf_corr = xr.align(chi_past_djf_corr, chi_nino34_past_djf_corr, join='inner')
chi_recent_djf_corr, chi_nino34_recent_djf_corr = xr.align(chi_recent_djf_corr, chi_nino34_recent_djf_corr, join='inner')

u_chi_clim_djf_corr, _ = xr.align(u_chi_clim_djf_corr, chi_nino34_clim_djf_corr, join='inner')
u_chi_past_djf_corr, _ = xr.align(u_chi_past_djf_corr, chi_nino34_past_djf_corr, join='inner')
u_chi_recent_djf_corr, _ = xr.align(u_chi_recent_djf_corr, chi_nino34_recent_djf_corr, join='inner')

v_chi_clim_djf_corr, _ = xr.align(v_chi_clim_djf_corr, chi_nino34_clim_djf_corr, join='inner')
v_chi_past_djf_corr, _ = xr.align(v_chi_past_djf_corr, chi_nino34_past_djf_corr, join='inner')
v_chi_recent_djf_corr, _ = xr.align(v_chi_recent_djf_corr, chi_nino34_recent_djf_corr, join='inner')

chi_corr_clim = xr.corr(chi_clim_djf_corr, chi_nino34_clim_djf_corr, dim='djf_year').compute()
chi_corr_past = xr.corr(chi_past_djf_corr, chi_nino34_past_djf_corr, dim='djf_year').compute()
chi_corr_recent = xr.corr(chi_recent_djf_corr, chi_nino34_recent_djf_corr, dim='djf_year').compute()
chi_corr_recent_minus_past = chi_corr_recent - chi_corr_past

u_chi_corr_clim = xr.corr(u_chi_clim_djf_corr, chi_nino34_clim_djf_corr, dim='djf_year').compute()
u_chi_corr_past = xr.corr(u_chi_past_djf_corr, chi_nino34_past_djf_corr, dim='djf_year').compute()
u_chi_corr_recent = xr.corr(u_chi_recent_djf_corr, chi_nino34_recent_djf_corr, dim='djf_year').compute()
u_chi_corr_recent_minus_past = u_chi_corr_recent - u_chi_corr_past

v_chi_corr_clim = xr.corr(v_chi_clim_djf_corr, chi_nino34_clim_djf_corr, dim='djf_year').compute()
v_chi_corr_past = xr.corr(v_chi_past_djf_corr, chi_nino34_past_djf_corr, dim='djf_year').compute()
v_chi_corr_recent = xr.corr(v_chi_recent_djf_corr, chi_nino34_recent_djf_corr, dim='djf_year').compute()
v_chi_corr_recent_minus_past = v_chi_corr_recent - v_chi_corr_past

chi_n_full = int(chi_clim_djf_corr.sizes['djf_year'])
chi_n_past = int(chi_past_djf_corr.sizes['djf_year'])
chi_n_recent = int(chi_recent_djf_corr.sizes['djf_year'])

chi_corr_clim_safe = xr.where(np.abs(chi_corr_clim) >= 0.999999, np.sign(chi_corr_clim) * 0.999999, chi_corr_clim)
chi_corr_past_safe = xr.where(np.abs(chi_corr_past) >= 0.999999, np.sign(chi_corr_past) * 0.999999, chi_corr_past)
chi_corr_recent_safe = xr.where(np.abs(chi_corr_recent) >= 0.999999, np.sign(chi_corr_recent) * 0.999999, chi_corr_recent)

chi_corr_clim_t = chi_corr_clim_safe * np.sqrt((chi_n_full - 2) / (1 - chi_corr_clim_safe ** 2))
chi_corr_past_t = chi_corr_past_safe * np.sqrt((chi_n_past - 2) / (1 - chi_corr_past_safe ** 2))
chi_corr_recent_t = chi_corr_recent_safe * np.sqrt((chi_n_recent - 2) / (1 - chi_corr_recent_safe ** 2))

chi_corr_clim_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=chi_n_full - 2), chi_corr_clim_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
chi_corr_past_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=chi_n_past - 2), chi_corr_past_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
chi_corr_recent_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=chi_n_recent - 2), chi_corr_recent_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()

chi_z_stat = (np.arctanh(chi_corr_recent_safe) - np.arctanh(chi_corr_past_safe)) / np.sqrt(1 / (chi_n_recent - 3) + 1 / (chi_n_past - 3))
chi_corr_recent_minus_past_p = xr.apply_ufunc(lambda x: 2 * norm.sf(np.abs(x)), chi_z_stat, vectorize=True, dask='allowed', output_dtypes=[float]).compute()

u_chi_corr_clim_safe = xr.where(np.abs(u_chi_corr_clim) >= 0.999999, np.sign(u_chi_corr_clim) * 0.999999, u_chi_corr_clim)
u_chi_corr_past_safe = xr.where(np.abs(u_chi_corr_past) >= 0.999999, np.sign(u_chi_corr_past) * 0.999999, u_chi_corr_past)
u_chi_corr_recent_safe = xr.where(np.abs(u_chi_corr_recent) >= 0.999999, np.sign(u_chi_corr_recent) * 0.999999, u_chi_corr_recent)
v_chi_corr_clim_safe = xr.where(np.abs(v_chi_corr_clim) >= 0.999999, np.sign(v_chi_corr_clim) * 0.999999, v_chi_corr_clim)
v_chi_corr_past_safe = xr.where(np.abs(v_chi_corr_past) >= 0.999999, np.sign(v_chi_corr_past) * 0.999999, v_chi_corr_past)
v_chi_corr_recent_safe = xr.where(np.abs(v_chi_corr_recent) >= 0.999999, np.sign(v_chi_corr_recent) * 0.999999, v_chi_corr_recent)

u_chi_corr_clim_t = u_chi_corr_clim_safe * np.sqrt((chi_n_full - 2) / (1 - u_chi_corr_clim_safe ** 2))
u_chi_corr_past_t = u_chi_corr_past_safe * np.sqrt((chi_n_past - 2) / (1 - u_chi_corr_past_safe ** 2))
u_chi_corr_recent_t = u_chi_corr_recent_safe * np.sqrt((chi_n_recent - 2) / (1 - u_chi_corr_recent_safe ** 2))
v_chi_corr_clim_t = v_chi_corr_clim_safe * np.sqrt((chi_n_full - 2) / (1 - v_chi_corr_clim_safe ** 2))
v_chi_corr_past_t = v_chi_corr_past_safe * np.sqrt((chi_n_past - 2) / (1 - v_chi_corr_past_safe ** 2))
v_chi_corr_recent_t = v_chi_corr_recent_safe * np.sqrt((chi_n_recent - 2) / (1 - v_chi_corr_recent_safe ** 2))

u_chi_corr_clim_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=chi_n_full - 2), u_chi_corr_clim_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
u_chi_corr_past_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=chi_n_past - 2), u_chi_corr_past_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
u_chi_corr_recent_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=chi_n_recent - 2), u_chi_corr_recent_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
v_chi_corr_clim_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=chi_n_full - 2), v_chi_corr_clim_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
v_chi_corr_past_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=chi_n_past - 2), v_chi_corr_past_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
v_chi_corr_recent_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=chi_n_recent - 2), v_chi_corr_recent_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()

chi_corr_clim_sig = chi_corr_clim_p < 0.05
chi_corr_past_sig = chi_corr_past_p < 0.05
chi_corr_recent_sig = chi_corr_recent_p < 0.05
chi_corr_recent_minus_past_sig = chi_corr_recent_minus_past_p < 0.05
chi_vector_sig_clim = (u_chi_corr_clim_p < 0.05) | (v_chi_corr_clim_p < 0.05)
chi_vector_sig_past = (u_chi_corr_past_p < 0.05) | (v_chi_corr_past_p < 0.05)
chi_vector_sig_recent = (u_chi_corr_recent_p < 0.05) | (v_chi_corr_recent_p < 0.05)
chi_vector_sig_recent_minus_past = (u_chi_corr_recent_p < 0.05) | (v_chi_corr_recent_p < 0.05)


In [ ]:
mslp_nino34_clim_djf_corr = nino34_clim_djf_corr.copy()
mslp_nino34_past_djf_corr = nino34_past_djf_corr.copy()
mslp_nino34_recent_djf_corr = nino34_recent_djf_corr.copy()

mslp_clim_djf_corr, mslp_nino34_clim_djf_corr = xr.align(mslp_clim_djf_corr, mslp_nino34_clim_djf_corr, join='inner')
mslp_past_djf_corr, mslp_nino34_past_djf_corr = xr.align(mslp_past_djf_corr, mslp_nino34_past_djf_corr, join='inner')
mslp_recent_djf_corr, mslp_nino34_recent_djf_corr = xr.align(mslp_recent_djf_corr, mslp_nino34_recent_djf_corr, join='inner')

mslp_corr_clim = xr.corr(mslp_clim_djf_corr, mslp_nino34_clim_djf_corr, dim='djf_year').compute()
mslp_corr_past = xr.corr(mslp_past_djf_corr, mslp_nino34_past_djf_corr, dim='djf_year').compute()
mslp_corr_recent = xr.corr(mslp_recent_djf_corr, mslp_nino34_recent_djf_corr, dim='djf_year').compute()
mslp_corr_recent_minus_past = mslp_corr_recent - mslp_corr_past

mslp_n_full = int(mslp_clim_djf_corr.sizes['djf_year'])
mslp_n_past = int(mslp_past_djf_corr.sizes['djf_year'])
mslp_n_recent = int(mslp_recent_djf_corr.sizes['djf_year'])

mslp_corr_clim_safe = xr.where(np.abs(mslp_corr_clim) >= 0.999999, np.sign(mslp_corr_clim) * 0.999999, mslp_corr_clim)
mslp_corr_past_safe = xr.where(np.abs(mslp_corr_past) >= 0.999999, np.sign(mslp_corr_past) * 0.999999, mslp_corr_past)
mslp_corr_recent_safe = xr.where(np.abs(mslp_corr_recent) >= 0.999999, np.sign(mslp_corr_recent) * 0.999999, mslp_corr_recent)

mslp_corr_clim_t = mslp_corr_clim_safe * np.sqrt((mslp_n_full - 2) / (1 - mslp_corr_clim_safe ** 2))
mslp_corr_past_t = mslp_corr_past_safe * np.sqrt((mslp_n_past - 2) / (1 - mslp_corr_past_safe ** 2))
mslp_corr_recent_t = mslp_corr_recent_safe * np.sqrt((mslp_n_recent - 2) / (1 - mslp_corr_recent_safe ** 2))

mslp_corr_clim_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=mslp_n_full - 2), mslp_corr_clim_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
mslp_corr_past_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=mslp_n_past - 2), mslp_corr_past_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
mslp_corr_recent_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=mslp_n_recent - 2), mslp_corr_recent_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()

mslp_z_stat = (np.arctanh(mslp_corr_recent_safe) - np.arctanh(mslp_corr_past_safe)) / np.sqrt(1 / (mslp_n_recent - 3) + 1 / (mslp_n_past - 3))
mslp_corr_recent_minus_past_p = xr.apply_ufunc(lambda x: 2 * norm.sf(np.abs(x)), mslp_z_stat, vectorize=True, dask='allowed', output_dtypes=[float]).compute()

mslp_corr_clim_sig = mslp_corr_clim_p < 0.05
mslp_corr_past_sig = mslp_corr_past_p < 0.05
mslp_corr_recent_sig = mslp_corr_recent_p < 0.05
mslp_corr_recent_minus_past_sig = mslp_corr_recent_minus_past_p < 0.05

print('MSLP DJF seasons:', int(mslp_clim_djf_corr.sizes['djf_year']))

In [ ]:
gh850_nino34_clim_djf_corr = nino34_clim_djf_corr.copy()
gh850_nino34_past_djf_corr = nino34_past_djf_corr.copy()
gh850_nino34_recent_djf_corr = nino34_recent_djf_corr.copy()

gh850_clim_djf_corr, gh850_nino34_clim_djf_corr = xr.align(gh850_clim_djf_corr, gh850_nino34_clim_djf_corr, join='inner')
gh850_past_djf_corr, gh850_nino34_past_djf_corr = xr.align(gh850_past_djf_corr, gh850_nino34_past_djf_corr, join='inner')
gh850_recent_djf_corr, gh850_nino34_recent_djf_corr = xr.align(gh850_recent_djf_corr, gh850_nino34_recent_djf_corr, join='inner')

gh850_corr_clim = xr.corr(gh850_clim_djf_corr, gh850_nino34_clim_djf_corr, dim='djf_year').compute()
gh850_corr_past = xr.corr(gh850_past_djf_corr, gh850_nino34_past_djf_corr, dim='djf_year').compute()
gh850_corr_recent = xr.corr(gh850_recent_djf_corr, gh850_nino34_recent_djf_corr, dim='djf_year').compute()
gh850_corr_recent_minus_past = gh850_corr_recent - gh850_corr_past

gh850_n_full = int(gh850_clim_djf_corr.sizes['djf_year'])
gh850_n_past = int(gh850_past_djf_corr.sizes['djf_year'])
gh850_n_recent = int(gh850_recent_djf_corr.sizes['djf_year'])

gh850_corr_clim_safe = xr.where(np.abs(gh850_corr_clim) >= 0.999999, np.sign(gh850_corr_clim) * 0.999999, gh850_corr_clim)
gh850_corr_past_safe = xr.where(np.abs(gh850_corr_past) >= 0.999999, np.sign(gh850_corr_past) * 0.999999, gh850_corr_past)
gh850_corr_recent_safe = xr.where(np.abs(gh850_corr_recent) >= 0.999999, np.sign(gh850_corr_recent) * 0.999999, gh850_corr_recent)

gh850_corr_clim_t = gh850_corr_clim_safe * np.sqrt((gh850_n_full - 2) / (1 - gh850_corr_clim_safe ** 2))
gh850_corr_past_t = gh850_corr_past_safe * np.sqrt((gh850_n_past - 2) / (1 - gh850_corr_past_safe ** 2))
gh850_corr_recent_t = gh850_corr_recent_safe * np.sqrt((gh850_n_recent - 2) / (1 - gh850_corr_recent_safe ** 2))

gh850_corr_clim_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=gh850_n_full - 2), gh850_corr_clim_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
gh850_corr_past_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=gh850_n_past - 2), gh850_corr_past_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()
gh850_corr_recent_p = xr.apply_ufunc(lambda x: 2 * student_t.sf(np.abs(x), df=gh850_n_recent - 2), gh850_corr_recent_t, vectorize=True, dask='allowed', output_dtypes=[float]).compute()

gh850_z_stat = (np.arctanh(gh850_corr_recent_safe) - np.arctanh(gh850_corr_past_safe)) / np.sqrt(1 / (gh850_n_recent - 3) + 1 / (gh850_n_past - 3))
gh850_corr_recent_minus_past_p = xr.apply_ufunc(lambda x: 2 * norm.sf(np.abs(x)), gh850_z_stat, vectorize=True, dask='allowed', output_dtypes=[float]).compute()

gh850_corr_clim_sig = gh850_corr_clim_p < 0.05
gh850_corr_past_sig = gh850_corr_past_p < 0.05
gh850_corr_recent_sig = gh850_corr_recent_p < 0.05
gh850_corr_recent_minus_past_sig = gh850_corr_recent_minus_past_p < 0.05

print('Geopotential height 850 hPa DJF seasons:', int(gh850_clim_djf_corr.sizes['djf_year']))

# 5. Plot

In [ ]:
corr_levels = np.arange(-1, 1.01, 0.05)
corr_ticks = np.arange(-1, 1.01, 0.25)
wind_step = 8
wind_scale = 30
wind_diff_scale = 15

RESULTS_DIR = Path('../../results').resolve() / 'analisis_korelasi_26-19'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def save_plot(fig, filename):
    fig.savefig(RESULTS_DIR / filename, dpi=300, bbox_inches='tight')


### Rainfall (scalar)

In [ ]:
plot_defs = [
    (rain_corr_clim, rain_corr_clim_sig, 'Korelasi Curah Hujan vs Nino34 DJF 1981-2025', 'rainfall_corr_1981-2025.png'),
    (rain_corr_past, rain_corr_past_sig, 'Korelasi Curah Hujan vs Nino34 DJF P1', 'rainfall_corr_p1.png'),
    (rain_corr_recent, rain_corr_recent_sig, 'Korelasi Curah Hujan vs Nino34 DJF P2', 'rainfall_corr_p2.png'),
    (rain_corr_recent_minus_past, rain_corr_recent_minus_past_sig, 'Selisih Korelasi Curah Hujan P2-P1', 'rainfall_corr_p2_minus_p1.png'),
]

for data, sig_mask, title, filename in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=corr_levels,
        extend='both',
        add_colorbar=False,
        infer_intervals=True,
    )

    sig_plot = (sig_mask.fillna(False).astype(np.int8).coarsen(lat=8, lon=8, boundary='trim').max() > 0)
    yy, xx = np.where(sig_plot.values)
    if yy.size > 0:
        ax.scatter(
            sig_plot['lon'].values[xx],
            sig_plot['lat'].values[yy],
            s=15,
            c='k',
            marker='.',
            linewidths=0,
            alpha=0.7,
            transform=ccrs.PlateCarree(),
            zorder=4,
        )

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=corr_ticks, spacing='proportional', extend='both')
    cbar.set_label('Koefisien korelasi', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)


### Wind (vector) - U Shade

In [ ]:
plot_defs = [
    (wind_u_corr_clim, wind_u_corr_clim, wind_v_corr_clim, 'Korelasi UWND vs Nino34 DJF 1981-2025', 'wind_u_corr_1981-2025.png', wind_scale, 1.0, 'corr = 1.0'),
    (wind_u_corr_past, wind_u_corr_past, wind_v_corr_past, 'Korelasi UWND vs Nino34 DJF P1', 'wind_u_corr_p1.png', wind_scale, 1.0, 'corr = 1.0'),
    (wind_u_corr_recent, wind_u_corr_recent, wind_v_corr_recent, 'Korelasi UWND vs Nino34 DJF P2', 'wind_u_corr_p2.png', wind_scale, 1.0, 'corr = 1.0'),
    (wind_u_corr_recent_minus_past, wind_u_corr_recent_minus_past, wind_v_corr_recent_minus_past, 'Selisih Korelasi UWND P2-P1', 'wind_u_corr_p2_minus_p1.png', wind_scale, 1.0, 'corr = 1.0'),
]

for shade_data, u_data, v_data, title, filename, scale, key_u, key_label in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = smooth_for_plot(shade_data)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=corr_levels,
        extend='both',
        add_colorbar=False,
        infer_intervals=True,
    )

    u_plot = u_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    v_plot = v_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    q = ax.quiver(
        u_plot['lon'].values,
        u_plot['lat'].values,
        u_plot.values,
        v_plot.values,
        transform=ccrs.PlateCarree(),
        scale=scale,
        width=0.002,
        color='black',
        zorder=4,
    )
    ax.quiverkey(q, X=0.88, Y=1.06, U=key_u, label=key_label, labelpos='E', coordinates='axes', color='black')

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=corr_ticks, spacing='proportional', extend='both')
    cbar.set_label('Koefisien korelasi u-wind', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)


### Wind (vector) - V Shade

In [ ]:
plot_defs = [
    (wind_v_corr_clim, wind_u_corr_clim, wind_v_corr_clim, 'Korelasi VWND vs Nino34 DJF 1981-2025', 'wind_v_corr_1981-2025.png', wind_scale, 1.0, 'corr = 1.0'),
    (wind_v_corr_past, wind_u_corr_past, wind_v_corr_past, 'Korelasi VWND vs Nino34 DJF P1', 'wind_v_corr_p1.png', wind_scale, 1.0, 'corr = 1.0'),
    (wind_v_corr_recent, wind_u_corr_recent, wind_v_corr_recent, 'Korelasi VWND vs Nino34 DJF P2', 'wind_v_corr_p2.png', wind_scale, 1.0, 'corr = 1.0'),
    (wind_v_corr_recent_minus_past, wind_u_corr_recent_minus_past, wind_v_corr_recent_minus_past, 'Selisih Korelasi VWND P2-P1', 'wind_v_corr_p2_minus_p1.png', wind_scale, 1.0, 'corr = 1.0'),
]

for shade_data, u_data, v_data, title, filename, scale, key_u, key_label in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = shade_data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=corr_levels,
        extend='both',
        add_colorbar=False,
        infer_intervals=True,
    )

    u_plot = u_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    v_plot = v_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    q = ax.quiver(
        u_plot['lon'].values,
        u_plot['lat'].values,
        u_plot.values,
        v_plot.values,
        transform=ccrs.PlateCarree(),
        scale=scale,
        width=0.002,
        color='black',
        zorder=4,
    )
    ax.quiverkey(q, X=0.88, Y=1.06, U=key_u, label=key_label, labelpos='E', coordinates='axes', color='black')

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=corr_ticks, spacing='proportional', extend='both')
    cbar.set_label('Koefisien korelasi v-wind', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)


### MFC (scalar + vector)

In [ ]:
plot_defs = [
    (mfc_corr_clim, mfc_qx_corr_clim, mfc_qy_corr_clim, 'Korelasi MFC vs Nino34 DJF 1981-2025', 'mfc_corr_1981-2025.png', wind_scale, 1.0, 'corr = 1.0'),
    (mfc_corr_past, mfc_qx_corr_past, mfc_qy_corr_past, 'Korelasi MFC vs Nino34 DJF P1', 'mfc_corr_p1.png', wind_scale, 1.0, 'corr = 1.0'),
    (mfc_corr_recent, mfc_qx_corr_recent, mfc_qy_corr_recent, 'Korelasi MFC vs Nino34 DJF P2', 'mfc_corr_p2.png', wind_scale, 1.0, 'corr = 1.0'),
    (mfc_corr_recent_minus_past, mfc_qx_corr_recent_minus_past, mfc_qy_corr_recent_minus_past, 'Selisih Korelasi MFC P2-P1', 'mfc_corr_p2_minus_p1.png', wind_scale, 1.0, 'corr = 1.0'),
]

for shade_data, u_data, v_data, title, filename, scale, key_u, key_label in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = smooth_for_plot(shade_data)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=corr_levels,
        extend='both',
        add_colorbar=False,
        infer_intervals=True,
    )

    u_plot = u_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    v_plot = v_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    q = ax.quiver(
        u_plot['lon'].values,
        u_plot['lat'].values,
        u_plot.values,
        v_plot.values,
        transform=ccrs.PlateCarree(),
        scale=scale,
        width=0.002,
        color='black',
        zorder=4,
    )
    ax.quiverkey(q, X=0.88, Y=1.06, U=key_u, label=key_label, labelpos='E', coordinates='axes', color='black')

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=corr_ticks, spacing='proportional', extend='both')
    cbar.set_label('Koefisien korelasi MFC', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)


### Streamfunction (scalar + vector)

In [ ]:
plot_defs = [
    (psi_corr_clim, u_psi_corr_clim, v_psi_corr_clim, 'Korelasi Streamfunction vs Nino34 DJF 1981-2025', 'streamfunction_corr_1981-2025.png', wind_scale, 1.0, 'corr = 1.0'),
    (psi_corr_past, u_psi_corr_past, v_psi_corr_past, 'Korelasi Streamfunction vs Nino34 DJF P1', 'streamfunction_corr_p1.png', wind_scale, 1.0, 'corr = 1.0'),
    (psi_corr_recent, u_psi_corr_recent, v_psi_corr_recent, 'Korelasi Streamfunction vs Nino34 DJF P2', 'streamfunction_corr_p2.png', wind_scale, 1.0, 'corr = 1.0'),
    (psi_corr_recent_minus_past, u_psi_corr_recent_minus_past, v_psi_corr_recent_minus_past, 'Selisih Korelasi Streamfunction P2-P1', 'streamfunction_corr_p2_minus_p1.png', wind_scale, 1.0, 'corr = 1.0'),
]

for shade_data, u_data, v_data, title, filename, scale, key_u, key_label in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = shade_data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=corr_levels,
        extend='both',
        add_colorbar=False,
        infer_intervals=True,
    )

    u_plot = u_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    v_plot = v_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    q = ax.quiver(
        u_plot['lon'].values,
        u_plot['lat'].values,
        u_plot.values,
        v_plot.values,
        transform=ccrs.PlateCarree(),
        scale=scale,
        width=0.002,
        color='black',
        zorder=4,
    )
    ax.quiverkey(q, X=0.88, Y=1.06, U=key_u, label=key_label, labelpos='E', coordinates='axes', color='black')

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=corr_ticks, spacing='proportional', extend='both')
    cbar.set_label('Koefisien korelasi streamfunction', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)


### Velocity potential (scalar + vector)

In [ ]:
plot_defs = [
    (chi_corr_clim, u_chi_corr_clim, v_chi_corr_clim, 'Korelasi Velocity Potential vs Nino34 DJF 1981-2025', 'velocity_potential_corr_1981-2025.png', wind_scale, 1.0, 'corr = 1.0'),
    (chi_corr_past, u_chi_corr_past, v_chi_corr_past, 'Korelasi Velocity Potential vs Nino34 DJF P1', 'velocity_potential_corr_p1.png', wind_scale, 1.0, 'corr = 1.0'),
    (chi_corr_recent, u_chi_corr_recent, v_chi_corr_recent, 'Korelasi Velocity Potential vs Nino34 DJF P2', 'velocity_potential_corr_p2.png', wind_scale, 1.0, 'corr = 1.0'),
    (chi_corr_recent_minus_past, u_chi_corr_recent_minus_past, v_chi_corr_recent_minus_past, 'Selisih Korelasi Velocity Potential P2-P1', 'velocity_potential_corr_p2_minus_p1.png', wind_scale, 1.0, 'corr = 1.0'),
]

for shade_data, u_data, v_data, title, filename, scale, key_u, key_label in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = shade_data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=corr_levels,
        extend='both',
        add_colorbar=False,
        infer_intervals=True,
    )

    u_plot = u_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    v_plot = v_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    q = ax.quiver(
        u_plot['lon'].values,
        u_plot['lat'].values,
        u_plot.values,
        v_plot.values,
        transform=ccrs.PlateCarree(),
        scale=scale,
        width=0.002,
        color='black',
        zorder=4,
    )
    ax.quiverkey(q, X=0.88, Y=1.06, U=key_u, label=key_label, labelpos='E', coordinates='axes', color='black')

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=corr_ticks, spacing='proportional', extend='both')
    cbar.set_label('Koefisien korelasi velocity potential', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)


### MSLP (scalar)

In [ ]:
plot_defs = [
    (mslp_corr_clim, mslp_corr_clim_sig, 'Korelasi MSLP vs Nino34 DJF 1981-2025', 'mslp_corr_1981-2025.png'),
    (mslp_corr_past, mslp_corr_past_sig, 'Korelasi MSLP vs Nino34 DJF P1', 'mslp_corr_p1.png'),
    (mslp_corr_recent, mslp_corr_recent_sig, 'Korelasi MSLP vs Nino34 DJF P2', 'mslp_corr_p2.png'),
    (mslp_corr_recent_minus_past, mslp_corr_recent_minus_past_sig, 'Selisih Korelasi MSLP P2-P1', 'mslp_corr_p2_minus_p1.png'),
]

for data, sig_mask, title, filename in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=corr_levels,
        extend='both',
        add_colorbar=False,
        infer_intervals=True,
    )

    sig_plot = (sig_mask.fillna(False).astype(np.int8).coarsen(lat=8, lon=8, boundary='trim').max() > 0)
    yy, xx = np.where(sig_plot.values)
    if yy.size > 0:
        ax.scatter(
            sig_plot['lon'].values[xx],
            sig_plot['lat'].values[yy],
            s=15,
            c='k',
            marker='.',
            linewidths=0,
            alpha=0.7,
            transform=ccrs.PlateCarree(),
            zorder=4,
        )

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=corr_ticks, spacing='proportional', extend='both')
    cbar.set_label('Koefisien korelasi MSLP', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)

### Geopotential Height 850 hPa (scalar)

In [ ]:
plot_defs = [
    (gh850_corr_clim, gh850_corr_clim_sig, 'Korelasi Geopotential Height 850 hPa vs Nino34 DJF 1981-2025', 'geopotential_height_850_corr_1981-2025.png'),
    (gh850_corr_past, gh850_corr_past_sig, 'Korelasi Geopotential Height 850 hPa vs Nino34 DJF P1', 'geopotential_height_850_corr_p1.png'),
    (gh850_corr_recent, gh850_corr_recent_sig, 'Korelasi Geopotential Height 850 hPa vs Nino34 DJF P2', 'geopotential_height_850_corr_p2.png'),
    (gh850_corr_recent_minus_past, gh850_corr_recent_minus_past_sig, 'Selisih Korelasi Geopotential Height 850 hPa P2-P1', 'geopotential_height_850_corr_p2_minus_p1.png'),
]

for data, sig_mask, title, filename in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=corr_levels,
        extend='both',
        add_colorbar=False,
        infer_intervals=True,
    )

    sig_plot = (sig_mask.fillna(False).astype(np.int8).coarsen(lat=8, lon=8, boundary='trim').max() > 0)
    yy, xx = np.where(sig_plot.values)
    if yy.size > 0:
        ax.scatter(
            sig_plot['lon'].values[xx],
            sig_plot['lat'].values[yy],
            s=15,
            c='k',
            marker='.',
            linewidths=0,
            alpha=0.7,
            transform=ccrs.PlateCarree(),
            zorder=4,
        )

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=corr_ticks, spacing='proportional', extend='both')
    cbar.set_label('Koefisien korelasi geopotential height 850 hPa', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)